In [1]:
import nest_asyncio 
nest_asyncio.apply()

import asyncio
import os

from pathlib import Path
from ollama import AsyncClient

from datetime import datetime


In [2]:
# Globally track or pass your target codebase directory
REPORT_TO_WRITE = "Monolith_Architecture_Report15a.md"
CODE_TO_ANALYZE = "/Users/cherylhoekstra/projects/grails-demo/"  # Set your path here
REPORT_DESC = "Framework Matrix:** Grails 3.x / Spring Boot / GORM / Redis"
MODEL_IN_USE = "qwen3:8b"  

# Output format: 2026-10-27 08:42:15
current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")


TARGET_EXTENSIONS = {'.java'}
# TARGET_EXTENSIONS = {'.groovy', '.java', '.yml', '.properties'}

EXCLUDE_DIRS = {'build', '.gradle', '.git', 'node_modules', '.gsp'}

MAX_STEPS = 5  # Protection loop ceiling - agentic AI using tools, in a loop


In [3]:
# [markdown]
# ### 1. Define the System Prompt & Tool Schemas

SYSTEM_PROMPT = """
Role: Senior Java/Groovy & Grails Monolith Systems Architect & Data Systems Auditor
Task: Analyze a hybrid Grails 3.x and Spring Boot monolithic application for structural layout, caching architecture, data flow bottlenecks, and Redis integration.

[INSTRUCTIONS FOR EXECUTION]1. You have tools to browse the codebase (`list_files`) and read specific file pieces (`read_file_chunk`).
2. Use these tools autonomously to find GORM domains, Spring services, and Redis configurations.
3. Keep track of what you discover. When you have found all relevant caching footprints and architectural details, output your final summary report.
4. DO NOT loop through every file if it is not relevant to caching or structural layout. Be efficient.

[FINAL OUTPUT FORMAT REQUIREMENT]
When you have completed your analysis and are ready to stop, output your assessment exactly in this format:

## 📊 Enterprise Redis Cache Catalog Summary
### 🔑 Identified Keys & Namespaces

| Namespace / Prefix | Estimated Type | Originating Component | Eviction Trigger |
| :--- | :--- | :--- | :--- |
| [e.g., `user:session:`] | [String/Hash] | [AuthService] | [Logout / Session Timeout] |

### ⚡ Caching Vulnerabilities & Failure Risks
- [List critical architectural issues like cross-container serialization gaps or GORM lazy-loading hooks]

# Detailed File Breakdown
### 📄 File: [File Path]
#### 🏛️ Core Artifact Mapping
- **Component Type:** [Grails Service / Spring Component / GORM Domain / Controller]
- **Lifecycle Management:** [Grails Lifecycle Managed / Spring Boot Container Managed]
#### 🔄 Dependencies & Data Flow
- **Inbound Injections:** [What beans/services are injected into this class]
- **Outbound Operations:** [What databases, services, or Redis keys does this class touch]
#### ⚡ Caching & Redis Footprint
- **Caching Mechanism:** [None / @Cacheable / Direct RedisTemplate]
- **Key Strategy:** [Provide the Redis key pattern used]
"""

# Schemas telling Ollama what tools are available
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "Returns a list of all relevant source code files inside the workspace directory.",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file_chunk",
            "description": "Reads and returns a specific syntax-aware text chunk of a code file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filepath": {"type": "string", "description": "The relative path to the file."},
                    "chunk_index": {"type": "integer", "description": "The 1-based index of the chunk to read."}
                },
                "required": ["filepath", "chunk_index"]
            }
        }
    }
]


In [4]:


# [markdown]
# ### 2. Implement the Codebase Management Tools (The Agent's "Hands")

MAX_CHARS_PER_CHUNK = 4000

#TARGET_EXTENSIONS = {'.groovy', '.java', '.yml', '.properties'}
#EXCLUDE_DIRS = {'build', '.gradle', '.git', 'node_modules', '.gsp'}

def split_code_file(code_content: str, max_chars: int = MAX_CHARS_PER_CHUNK) -> list[str]:
    """Syntax-aware file-splitter separating blocks at top-level brace boundaries."""
    
    print(f"Splitting code content into chunks with max {max_chars} characters each.")
    
    lines = code_content.splitlines()
    chunks = []
    current_chunk = []
    current_length = 0
    brace_count = 0
    
    for line in lines:
        stripped = line.strip()
        current_chunk.append(line)
        current_length += len(line) + 1
        brace_count += stripped.count('{')
        brace_count -= stripped.count('}')
       
        if current_length >= max_chars and brace_count <= 0:
            chunks.append("\n".join(current_chunk))
            current_chunk = []
            current_length = 0
            brace_count = 0
            
    if current_chunk:
        chunks.append("\n".join(current_chunk))
    return chunks




In [5]:

def list_files(directory: str, pattern: str = None) -> list[str]:
    """Scans the designated target directory for matching codebase files."""
    
    print(f"Starting file scan in root directory: {CODE_TO_ANALYZE}")
    
    found_files = []
    base_path = Path(CODE_TO_ANALYZE).resolve()
    
    if not base_path.exists():
        return json.dumps({"error": f"Root directory '{CODE_TO_ANALYZE}' does not exist."})

    for root, dirs, files in os.walk(base_path):
        
        print(f"Scanning directory: {root} with {len(files)} files.")
        
        # Filter out excluded directories in-place
        dirs[:] = [d for d in dirs if d not in EXCLUDE_DIRS]
        
        for file in files:
            
            # print(f"Checking file: {file}") 
            
         
            if Path(file).suffix in TARGET_EXTENSIONS:
                # Keep paths relative to the codebase root so the agent sees clean paths
                full_path = os.path.join(root, file)
                relative_path = os.path.relpath(full_path, base_path)
                found_files.append(relative_path)
                
    print(f"Found {len(found_files)} relevant files in the codebase.")            
    return json.dumps({"files": sorted(found_files)})




In [6]:
def read_file_chunk(filepath: str, chunk_index: int) -> str:
    """Reads a chunk from a file relative to the codebase root directory."""
    
    print(f"Attempting to read chunk {chunk_index} from file: {filepath}")
    
    try:
        # Resolve the path safely against the codebase root
        base_path = Path(CODE_TO_ANALYZE).resolve()
        path = (base_path / filepath).resolve()
        
        # Security check: Ensure the agent isn't trying to escape the codebase directory
        if not path.is_relative_to(base_path):
            return json.dumps({"error": "Access denied: Path is outside the codebase root."})
            
        if not path.exists():
            return json.dumps({"error": f"File {filepath} not found."})
            
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            
        chunks = split_code_file(content)
        if chunk_index < 1 or chunk_index > len(chunks):
            return json.dumps({"error": f"Invalid chunk index. File has {len(chunks)} chunks."})
            
        return json.dumps({
            "filepath": filepath,
            "chunk_index": chunk_index,
            "total_chunks": len(chunks),
            "content": chunks[chunk_index - 1]
        })
    except Exception as e:
        return json.dumps({"error": str(e)})

In [7]:
TOOL_MAP = {
    "list_files": list_files,
    "read_file_chunk": read_file_chunk
}


In [8]:
import json

def extract_tool_calls(agent_decision_stream):
    # Check if Ollama parsed it natively into the object
    if agent_decision_stream.message.tool_calls:
        print(f"tool_calls: {agent_decision_stream.message.tool_calls}")
        return agent_decision_stream.message.tool_calls
        
    # Fallback Check: Did it leak as a text string inside 'content'?
    content = agent_decision_stream.message.content.strip()
    content_stripped = content.strip()
    
    print(f"in extract content: {content_stripped}")
    
    if content_stripped.startswith('{') and 'name' in content_stripped:
        try:
            # Structuring raw text JSON to mimic standard ToolCall format
            parsed = json.loads(content_stripped)
            return [{
                'function': {
                    'name': parsed.get('name'),
                    'arguments': parsed.get('arguments', {})
                }
            }]
        except json.JSONDecodeError:
            pass # Not a valid JSON tool call string, treat as conversational content
            
    return []



In [9]:

# [markdown]
# ### 3. The Autonomous Agent Framework Loop (The Agent's "Brain")

from urllib import response

async def run_autonomous_architect_agent(model_name="qwen2.5-coder:14b"):
    client = AsyncClient()
    
    # The short-term execution memory of your Agent
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Analyze our monolith codebase. Find the layout, dependency footprints, and map all Redis configuration strategies."}
    ]
    
    # max_steps = 15  # Protection loop ceiling - agentic AI using tools, in a loop
    
    print("🦉 Initializing Autonomous Systems Architect Agent...")
    
    for step in range(MAX_STEPS):
        print(f"\n--- 🦕 [Step {step + 1}/{MAX_STEPS}] Agent Reasoning Loop ---")
        
        print(f"\nCODE_TO_ANALYZE is: {CODE_TO_ANALYZE}")
        
        # Request a decision stream from Ollama
        agent_decision_stream = await client.chat(
            model=model_name,
            messages=messages,
            tools=[list_files, read_file_chunk],
            #tools=TOOLS_SCHEMA
        )

        if hasattr(agent_decision_stream, 'model_dump_json'):
            print(agent_decision_stream.model_dump_json(indent=2))
        else:
            # Fallback if it's somehow already parsed into a dictionary
            print(json.dumps(agent_decision_stream, indent=2))
        
        # print(f"\nagent_message_role: {agent_decision_stream.message.role}")
        
        # messages.append(agent_decision_stream.message)
        # print(f"\nmessages.append ... to a list of dicts: {messages.append}")
        
        # Convert to dict safely if it's an Ollama/OpenAI response object (dumped in a string)
        # Check if the object supports Pydantic v2 serialization
        if hasattr(agent_decision_stream.message, 'model_dump'):
            agent_decision_stream_dict = agent_decision_stream.message.model_dump()
            print(f"\nobject supports Pydantic v2 serialization: {agent_decision_stream_dict}")
        elif hasattr(agent_decision_stream.message, 'message'):
            agent_decision_stream_dict = agent_decision_stream.message.message
            print(f"\nobject is an older Pydantic v1 object or a plain string: {agent_decision_stream_dict}")
        else:
            agent_decision_stream_dict = agent_decision_stream.message
            print(f"\nobject is an older Pydantic v1 object or a plain string: {agent_decision_stream_dict}")

        # Check both existence AND that it isn't an empty array
        if agent_decision_stream_dict.get('tool_calls'):
            print("\nFound active tool calls")
        else:
            print("\nModel chose to reply with text or tool definitions are missing.")


        # Usage in your Orchestrator Loop:
        active_tool_calls = extract_tool_calls(agent_decision_stream)
        
        if active_tool_calls:
            for tool_call in agent_decision_stream.message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = tool_call.function.arguments
                print(f"Found Tool Call: {tool_name}")
                if tool_name == "list_files":
                    directory = tool_args.get("directory")
                    pattern = tool_args.get("file_pattern")
                    print(f"   -> Scanning: '{directory}' matching pattern from tool_args: '{pattern}'")
                elif tool_name == "read_file_chunk":
                    filepath = tool_args.get("filepath")
                    print(f"   -> Reading chunk from: {filepath}")
            
        else:
            print(f"No tools were called. Model message: {agent_decision_stream.message.content}")
        
        if agent_decision_stream.message.get("tool_calls"):
            
            print(f"\nin agent_message tool_calls...")
            
            for tool_call in agent_decision_stream.message['tool_calls']:
                
                func_name = tool_call.function.name
                          
                func_args = tool_call.function.arguments # Ollama supplies args dynamically
                # func_args = tool_call['function']['arguments'] 
                func_args_json = json.dumps(func_args, indent=2)
                
                print(f"\n🦆 [Action] Agent called tool: \n{func_name}")
                print(f"func_args_json: {func_args_json}")
                
                # Execute the bound tool
                if func_name in TOOL_MAP:
                        
                    print(f"\nfunc_name in TOOL_MAP: {func_name}, args: {func_args}")
                    # print(f"\nffunc_args: {func_args_json}")
                    
                    # print(f"\nisinstance(func_args, str): {isinstance(func_args, str)}")
                    
                    # Handle json conversions if arguments arrive as a string
                    if isinstance(func_args, str):
                        func_args = json.loads(func_args)
                          
                    tool_result = TOOL_MAP[func_name](**func_args)
                    
                else:
                    tool_result = f"\nError: Tool '{func_name}' is not supported."
                    print(f"ttool_result: {tool_result}")
                
                print(f"\n[Observation] Tool responded with payload content.")
                
                # Feed the raw observation context back into the conversation thread
                messages.append({
                    "role": "tool",
                    "name": func_name,
                    "content": tool_result
                 })
        else:
            # If no tools are designated, the agent has finished its work and generated the markdown report
            print("\n✅ Agent Evaluation Completed Successfully!")
            print(agent_decision_stream.message['content'])
            
            # Write final agent product down directly to your report file
            with open(REPORT_TO_WRITE, "w", encoding="utf-8") as out_f:
                
                out_f.write(f"**Generated on:** {current_time}\n\n")  
                out_f.write(f"**Code base:** {CODE_TO_ANALYZE}\n\n")
                out_f.write(f"**Framework Matrix:** {REPORT_DESC}\n\n")
                out_f.write(f"**Model used:** {MODEL_IN_USE}\n\n")
                
                out_f.write(agent_decision_stream.message['content'])
            return
            
    print("\n❌ System Alert: Agent aborted execution. Max logical steps exceeded.")


# Trigger the asynchronous main executor loop
if __name__ == "__main__":
    asyncio.run(run_autonomous_architect_agent(MODEL_IN_USE))
    


🦉 Initializing Autonomous Systems Architect Agent...

--- 🦕 [Step 1/5] Agent Reasoning Loop ---

CODE_TO_ANALYZE is: /Users/cherylhoekstra/projects/grails-demo/
{
  "model": "qwen3:8b",
  "created_at": "2026-10-28T06:24:13.812243Z",
  "done": true,
  "done_reason": "stop",
  "total_duration": 51338804583,
  "load_duration": 2061328375,
  "prompt_eval_count": 659,
  "prompt_eval_duration": 3393833000,
  "eval_count": 800,
  "eval_duration": 45873629000,
  "message": {
    "role": "assistant",
    "content": "",
    "thinking": "Okay, let's start by understanding the user's request. They want me to analyze their monolith codebase, focusing on the layout, dependency footprints, and Redis configurations. The tools I have are list_files and read_file_chunk. \n\nFirst, I need to find all the relevant files. Since the user mentioned Grails 3.x and Spring Boot, I should look for directories typical in such projects. Common places for GORM domains would be under src/main/groovy, and services mi